# Fold-local bull/reentry participation target v1

- Baselines: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct failed source `cap10_wf2020_failed_fold_objective_audit_v1`.
- Role: diagnose/replace target semantics before any retained actor training.
- Target window: `wf_2020_train` target contract from the SOXLBacktest qlib buy/sell conversion.
- Changed layer: target labels only. No actor retention or promotion is allowed from this card.
- External reference: SOXLBacktest buy/sell path is used as a timing baseline; low-risk variants remain drawdown references, not retained models.
- Acceptance: lift fold-local train-window bull/reentry participation rows, preserve original risk targets, and prove no eval/downtrend/drawdown-control leakage.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src/soxl_quant").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.soxl_quant.app.state_objective_training_targets import (
    BullReentryParticipationTargetConfig,
    apply_bull_reentry_participation_target_gate,
    audit_state_objective_target_lift,
    summarize_state_objective_target_lift_audit,
)

CARD = ROOT / "runtime/model_selection/card/fold_local_bull_reentry_participation_target_v1"
SOURCE_CONTRACT = ROOT / "runtime/model_selection/card/cap10_wf2020_soxlbacktest_target_contract_v1/soxlbacktest_wf2020_target_contract.csv"
FAILED_AUDIT = ROOT / "runtime/model_selection/card/cap10_wf2020_failed_fold_objective_audit_v1/summary.json"
STAGE_METRICS = ROOT / "runtime/model_selection/card/cap10_wf2020_failed_fold_objective_audit_v1/soxlbacktest_wf2020_stage_metrics.csv"
READINESS = ROOT / "runtime/model_selection/card/retention_frontier_target_revision_readiness_v1/summary.json"

targets = pd.read_csv(SOURCE_CONTRACT)
config = BullReentryParticipationTargetConfig(
    enabled=True,
    min_target_position=0.70,
    max_target_position=0.85,
    min_lift=0.01,
    objective_weight_bonus=5.0,
    max_objective_weight=30.0,
    include_release_gate=True,
    forward_hold_days=10,
    exclude_downtrend=True,
    exclude_drawdown_control=True,
)

revised = apply_bull_reentry_participation_target_gate(targets, config)

def _safe(value):
    if hasattr(value, "item"):
        value = value.item()
    if isinstance(value, float) and pd.isna(value):
        return None
    return value

def _position_audit(label: str, frame: pd.DataFrame) -> dict[str, object]:
    dates = pd.to_datetime(frame["datetime"])
    risk = pd.to_numeric(frame["risk_target_position"], errors="coerce")
    target_state = frame.get("bull_reentry_target_state", pd.Series(False, index=frame.index)).fillna(False).astype(bool)
    forward_state = frame.get("bull_reentry_forward_ready", pd.Series(False, index=frame.index)).fillna(False).astype(bool)
    train_ready = frame.get("bull_reentry_train_window_ready", pd.Series(True, index=frame.index)).fillna(True).astype(bool)
    downtrend = frame.get("is_downtrend_state", pd.Series(False, index=frame.index)).fillna(False).astype(bool)
    drawdown = frame.get("drawdown_control_state", pd.Series(False, index=frame.index)).fillna(False).astype(bool)
    release = frame.get("release_gate_state", pd.Series(False, index=frame.index)).fillna(False).astype(bool)
    return {
        "contract": label,
        "fold": ",".join(sorted(frame.get("fold", pd.Series([""])).dropna().astype(str).unique())),
        "rows": int(len(frame)),
        "start": dates.min().strftime("%Y-%m-%d"),
        "end": dates.max().strftime("%Y-%m-%d"),
        "mean_risk_target_position": float(risk.mean()),
        "median_risk_target_position": float(risk.median()),
        "p90_risk_target_position": float(risk.quantile(0.90)),
        "max_risk_target_position": float(risk.max()),
        "rows_ge_50pct": int((risk >= 0.50).sum()),
        "rows_ge_70pct": int((risk >= 0.70).sum()),
        "share_ge_70pct": float((risk >= 0.70).mean()),
        "release_rows": int(release.sum()),
        "bull_reentry_target_rows": int(target_state.sum()),
        "bull_reentry_forward_rows": int((target_state & forward_state).sum()),
        "eval_leakage_rows": int((target_state & ~train_ready).sum()),
        "downtrend_lifted_rows": int((target_state & downtrend).sum()),
        "drawdown_control_lifted_rows": int((target_state & drawdown).sum()),
    }

before_audit = _position_audit("source_contract", targets)
after_audit = _position_audit("bull_reentry_revision", revised)
position_audit = pd.DataFrame([before_audit, after_audit])
lift_audit = audit_state_objective_target_lift(
    revised,
    group_cols=("fold",),
    baseline_risk_position_col="risk_target_position_original",
)
lift_summary = summarize_state_objective_target_lift_audit(lift_audit)
action_state_counts = (
    revised.assign(
        action=revised.get("action", "").fillna(""),
        bull_reentry_target_state=revised["bull_reentry_target_state"].fillna(False).astype(bool),
        bull_reentry_forward_ready=revised["bull_reentry_forward_ready"].fillna(False).astype(bool),
    )
    .groupby(["action", "bull_reentry_target_state", "bull_reentry_forward_ready"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["bull_reentry_target_state", "rows", "action"], ascending=[False, False, True])
)

stage_metrics = pd.read_csv(STAGE_METRICS) if STAGE_METRICS.exists() else pd.DataFrame()
legacy_rebound = stage_metrics[(stage_metrics.get("candidate") == "legacy_default") & (stage_metrics.get("stage") == "rebound")]
legacy_full = stage_metrics[(stage_metrics.get("candidate") == "legacy_default") & (stage_metrics.get("stage") == "full_year")]
readiness = json.loads(READINESS.read_text(encoding="utf-8")) if READINESS.exists() else {}
failed_audit = json.loads(FAILED_AUDIT.read_text(encoding="utf-8")) if FAILED_AUDIT.exists() else {}

decision = "target_revision_ready_for_three_fold_bc_checkpoint_metric_replay"
if after_audit["bull_reentry_target_rows"] <= 0:
    decision = "target_revision_failed_no_bull_reentry_rows"
elif after_audit["eval_leakage_rows"] or after_audit["downtrend_lifted_rows"] or after_audit["drawdown_control_lifted_rows"]:
    decision = "target_revision_failed_leakage_or_risk_guard_violation"

summary = {
    "experiment": "fold_local_bull_reentry_participation_target_v1",
    "created_at": pd.Timestamp.utcnow().isoformat(),
    "decision": decision,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "cap10_wf2020_failed_fold_objective_audit_v1",
    "source_card": "cap10_wf2020_failed_fold_objective_audit_v1",
    "changed_layer": "target",
    "role": "diagnose_replace_target_semantics",
    "target_contract_gate": decision == "target_revision_ready_for_three_fold_bc_checkpoint_metric_replay",
    "bc_readiness_gate": decision == "target_revision_ready_for_three_fold_bc_checkpoint_metric_replay",
    "bc_screen_allowed_next": decision == "target_revision_ready_for_three_fold_bc_checkpoint_metric_replay",
    "source_contract": str(SOURCE_CONTRACT.relative_to(ROOT)),
    "config": config.__dict__,
    "source_rows_ge_70pct": before_audit["rows_ge_70pct"],
    "source_share_ge_70pct": before_audit["share_ge_70pct"],
    "revised_rows_ge_70pct": after_audit["rows_ge_70pct"],
    "revised_share_ge_70pct": after_audit["share_ge_70pct"],
    "bull_reentry_target_rows": after_audit["bull_reentry_target_rows"],
    "bull_reentry_forward_rows": after_audit["bull_reentry_forward_rows"],
    "eval_leakage_rows": after_audit["eval_leakage_rows"],
    "downtrend_lifted_rows": after_audit["downtrend_lifted_rows"],
    "drawdown_control_lifted_rows": after_audit["drawdown_control_lifted_rows"],
    "lift_summary": {key: _safe(value) for key, value in lift_summary.items()},
    "soxlbacktest_reference_role": "buy_sell_timing_baseline_not_retention_model",
    "soxlbacktest_legacy_rebound_return_pct": _safe(legacy_rebound["strategy_return_pct"].iloc[0]) if not legacy_rebound.empty else None,
    "soxlbacktest_legacy_rebound_avg_position": _safe(legacy_rebound["avg_position"].iloc[0]) if not legacy_rebound.empty else None,
    "soxlbacktest_legacy_full_return_pct": _safe(legacy_full["strategy_return_pct"].iloc[0]) if not legacy_full.empty else None,
    "soxlbacktest_legacy_full_max_dd_pct": _safe(legacy_full["strategy_max_dd_pct"].iloc[0]) if not legacy_full.empty else None,
    "dashboard_replay_ready_count": readiness.get("dashboard_replay_ready_count"),
    "dashboard_replay_profile_count": readiness.get("dashboard_replay_profile_count"),
    "auto_update_configured": readiness.get("auto_update_configured"),
    "auto_update_loaded_task_count": readiness.get("auto_update_loaded_task_count"),
    "next_action": "rerun_three_fold_bc_checkpoint_metric_replay_with_bull_reentry_target_contract",
    "next_mechanism": "rerun_three_fold_bc_checkpoint_metric_replay_with_bull_reentry_target_contract",
    "required_controls": failed_audit.get("required_next_run_controls", []),
    "sidecars": {
        "target_contract": "bull_reentry_target_contract.csv",
        "position_audit": "target_position_audit.csv",
        "target_lift_audit": "target_lift_audit.csv",
        "action_state_counts": "action_state_counts.csv",
        "summary": "summary.json",
    },
}

revised.to_csv(CARD / "bull_reentry_target_contract.csv", index=False)
position_audit.to_csv(CARD / "target_position_audit.csv", index=False)
lift_audit.to_csv(CARD / "target_lift_audit.csv", index=False)
action_state_counts.to_csv(CARD / "action_state_counts.csv", index=False)
(CARD / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "experiment": "fold_local_bull_reentry_participation_target_v1",
  "created_at": "2026-06-13T23:43:17.111141+00:00",
  "decision": "target_revision_ready_for_three_fold_bc_checkpoint_metric_replay",
  "promotion_allowed": false,
  "actor_retention_allowed": false,
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "cap10_wf2020_failed_fold_objective_audit_v1",
  "source_card": "cap10_wf2020_failed_fold_objective_audit_v1",
  "changed_layer": "target",
  "role": "diagnose_replace_target_semantics",
  "target_contract_gate": true,
  "bc_readiness_gate": true,
  "bc_screen_allowed_next": true,
  "source_contract": "runtime/model_selection/card/cap10_wf2020_soxlbacktest_target_contract_v1/soxlbacktest_wf2020_target_contract.csv",
  "config": {
    "enabled": true,
    "min_target_position": 0.7,
    "max_target_position": 0.85,
    "min_lift": 0.01,
    "objective_weight_bonus": 5.0,
    "max_objective_weight": 30.0,
    "include_release_ga

## Conclusion

This card is a target-generation handoff, not a retained model. If the executed `summary.json` decision is `target_revision_ready_for_three_fold_bc_checkpoint_metric_replay`, the next valid step is rerunning the three-fold BC/checkpoint metric replay with the generated `bull_reentry_target_contract.csv`. Actor retention remains blocked until the rerun passes the walk-forward drawdown and return gates.